<a href="https://colab.research.google.com/github/Chalhotra/ViT-Token-Economy/blob/pipeline%2Fchetak/notebooks/01_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baselines: ViT-Tiny and DeiT-Tiny (ImageNet-100)

This notebook is intentionally thin: it calls into `src/` modules.

In [5]:
import getpass

token = getpass.getpass("Enter GitHub token: ")

!git clone https://{token}@github.com/Chalhotra/ViT-Token-Economy.git
%cd ViT-Token-Economy

Enter GitHub token: ··········
Cloning into 'ViT-Token-Economy'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 22 (delta 0), reused 22 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 29.50 KiB | 9.83 MiB/s, done.
/content/ViT-Token-Economy


In [8]:
!git checkout pipeline/chetak

Branch 'pipeline/chetak' set up to track remote branch 'pipeline/chetak' from 'origin'.
Switched to a new branch 'pipeline/chetak'


In [15]:
!pip -q install -r requirements.txt
!pip -q install -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vit-deit-baselines (pyproject.toml) ... done


In [11]:
from src.imagenet_mapping import build_imagenet100_to_1k_map
from src.models import ModelConfig, create_model, shrink_imagenet1k_head_to_imagenet100
from src.data import DataConfig, load_imagenet100_split, build_transform_for_model, apply_timm_preprocess, build_loader
from src.eval import evaluate_accuracy_latency_throughput, compute_gflops
from src.utils import get_device, num_params
import torch

In [12]:
device = get_device()
maps = build_imagenet100_to_1k_map()

In [13]:
def run(model_id: str, batch_size: int = 64):
    model = create_model(ModelConfig(model_id=model_id, pretrained=True))
    model = shrink_imagenet1k_head_to_imagenet100(model, maps.new_to_old_map, num_classes=100)
    model = model.to(device).eval()
    ds = load_imagenet100_split(DataConfig(split='validation'))
    transform = build_transform_for_model(model)
    ds_t = apply_timm_preprocess(ds, transform)
    loader = build_loader(ds_t, DataConfig(batch_size=batch_size, split='validation', shuffle=False))
    metrics = evaluate_accuracy_latency_throughput(model, loader, device)
    sample = ds_t[0]['pixel_values'].unsqueeze(0).to(device)
    gflops = compute_gflops(model, sample)
    return {
        'model': model_id,
        'params_m': num_params(model)/1e6,
        'gflops': gflops,
        **metrics
    }

In [ ]:
run('vit_tiny_patch16_224')

In [ ]:
run('deit_tiny_patch16_224')